In [1]:
import sqlite3

conn = sqlite3.connect("law.db")
cursor = conn.cursor()

In [22]:
conn.close()

In [2]:
def extract_from_db(id: int, include_parent=False):
    cursor.execute("SELECT * FROM laws WHERE id = ?", (id,))
    rows = cursor.fetchall()

    columns = [col[0] for col in cursor.description]
    results = [dict(zip(columns, row)) for row in rows]

    if include_parent:
        all_nodes = []
        for r in results:
            current = r
            while current['parent_id'] is not None:
                cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
                parent = cursor.fetchone()
                if parent is None:
                    break
                parent_dict = dict(zip(columns, parent))
                all_nodes.append(parent_dict)
                current = parent_dict
        results.extend(all_nodes)

    return results[::-1]

In [7]:
rows = extract_from_db(166, include_parent=True)
law = ""
for r in rows:
    title = r['title']
    if not (title.strip().startswith("Chương") or title.strip().startswith("dâda")):
        law += r['content'] + "\n"
print(law)

Vượt xe và nhường đường cho xe xin vượt
Không được vượt xe trong trường hợp sau đây:
Trên cầu hẹp có một làn đường;



In [5]:
import py_vncorenlp
if "rdrsegmenter" not in globals():
    import py_vncorenlp
    rdrsegmenter = py_vncorenlp.VnCoreNLP(
        annotators=["wseg", "pos", "ner"],
        save_dir=r"E:\Github\LawAssistant\test\VnCoreNLP-master"
    )

In [ ]:
rdrsegmenter.close()

In [8]:
import re
import stopwordsiso as stopwords

stop_words = stopwords.stopwords("vi")

def clean_text(text: str) -> str:
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    return text.strip().lower()

text = """
Vượt xe và nhường đường cho xe xin vượt
Không được vượt xe trong trường hợp sau đây:
Trên cầu hẹp có một làn đường;
"""
text = clean_text(text)
print(text)

#Annotate original text (keep full segmentation)
output = rdrsegmenter.annotate_text(text)

pos_map = {
    "R": "Adverb (Trạng từ)",
    "V": "Verb (Động từ)",
    "N": "Noun (Danh từ)",
    "E": "Preposition (Giới từ)",
    "P": "Pronoun (Đại từ)",
    "CH": "Punctuation (Dấu câu)",
    "A": "Adjective (Tính từ)",
    "M": "Numeral (Số từ)"
}

print(f"{'Idx':<5} {'Token':<15} {'POS':<20} {'NER':<5}")
print("-"*50)

#Print and filter stopwords AFTER annotation
filtered_output = []
for sent in output.values() if isinstance(output, dict) else output:
    filtered_sent = []
    for token in sent:
        if token["wordForm"] not in stop_words:
            pos_full = pos_map.get(token["posTag"], token["posTag"])
            print(f"{token['index']:<5} {token['wordForm']:<15} {pos_full:<20} {token['nerLabel']:<5}")
            filtered_sent.append(token)
    filtered_output.append(filtered_sent)

#Extract noun-based concepts from filtered tokens
concepts = []
for sent in filtered_output:
    current = []
    for word in sent:
        token, pos = word["wordForm"], word["posTag"]
        if pos.startswith("N"):
            current.append(token)
        else:
            if current:
                concepts.append(" ".join(current))
                current = []
    if current:
        concepts.append(" ".join(current))

print("\nConcepts extracted:")
for c in concepts:
    print("-", c)

vượt xe và nhường đường cho xe xin vượt không được vượt xe trong trường hợp sau đây: trên cầu hẹp có một làn đường;
Idx   Token           POS                  NER  
--------------------------------------------------
1     vượt            Verb (Động từ)       O    
2     xe              Noun (Danh từ)       O    
4     nhường          Verb (Động từ)       O    
5     đường           Noun (Danh từ)       O    
7     xe              Noun (Danh từ)       O    
8     xin             Verb (Động từ)       O    
9     vượt            Verb (Động từ)       O    
12    vượt            Verb (Động từ)       O    
13    xe              Noun (Danh từ)       O    
15    trường_hợp      Noun (Danh từ)       O    
18    :               Punctuation (Dấu câu) O    
20    cầu             Noun (Danh từ)       O    
21    hẹp             Adjective (Tính từ)  O    
24    làn_đường       Noun (Danh từ)       O    
25    ;               Punctuation (Dấu câu) O    

Concepts extracted:
- xe
- đường xe
- xe trườn

In [55]:
import re
import stopwordsiso as stopwords

stop_words = stopwords.stopwords("vi")

def clean_text(text: str) -> str:
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    return text.strip().lower()

# -------------------------------
# 1. Original text
# -------------------------------
text = "Không được vượt xe trong trường hợp sau đây: Trên cầu hẹp có một làn đường;"
print("1. Original text:")
print(text, "\n")

# -------------------------------
# 2. Cleaned text
# -------------------------------
text = clean_text(text)
print("2. Cleaned text:")
print(text, "\n")

# -------------------------------
# 3. Word segmentation
# -------------------------------
segmented = rdrsegmenter.word_segment(text)
print("3. Segmented text (tokens):")
print(segmented, "\n")

# -------------------------------
# 4. POS + NER annotation
# -------------------------------
output = rdrsegmenter.annotate_text(text)

pos_map = {
    "R": "Adverb (Trạng từ)",
    "V": "Verb (Động từ)",
    "N": "Noun (Danh từ)",
    "E": "Preposition (Giới từ)",
    "P": "Pronoun (Đại từ)",
    "CH": "Punctuation (Dấu câu)",
    "A": "Adjective (Tính từ)",
    "M": "Numeral (Số từ)"
}

print("4. POS + NER annotation:")
print(f"{'Idx':<5} {'Token':<15} {'POS':<20} {'NER':<5}")
print("-"*55)

for sent in output.values() if isinstance(output, dict) else output:
    for token in sent:
        pos_full = pos_map.get(token["posTag"], token["posTag"])
        print(f"{token['index']:<5} {token['wordForm']:<15} {pos_full:<20} {token['nerLabel']:<5}")
print()

# -------------------------------
# 5. Stopword filtering
# -------------------------------
print("5. Tokens after stopword removal:")
filtered_output = []
for sent in output.values() if isinstance(output, dict) else output:
    filtered_sent = [t for t in sent if t["wordForm"] not in stop_words]
    filtered_output.append(filtered_sent)

for sent in filtered_output:
    print([t["wordForm"] for t in sent])
print()

# -------------------------------
# 6. Concept extraction (Nouns only)
# -------------------------------
print("6. Extracted concepts (noun phrases):")
concepts = []
for sent in filtered_output:
    current = []
    for word in sent:
        token, pos = word["wordForm"], word["posTag"]
        if pos.startswith("N"):  # noun or compound noun
            current.append(token)
        else:
            if current:
                concepts.append(" ".join(current))
                current = []
    if current:
        concepts.append(" ".join(current))

for c in concepts:
    print("-", c)


1. Original text:
Không được vượt xe trong trường hợp sau đây: Trên cầu hẹp có một làn đường; 

2. Cleaned text:
không được vượt xe trong trường hợp sau đây: trên cầu hẹp có một làn đường; 

3. Segmented text (tokens):
['không được vượt xe trong trường_hợp sau đây : trên cầu hẹp có một làn_đường ;'] 

4. POS + NER annotation:
Idx   Token           POS                  NER  
-------------------------------------------------------
1     không           Adverb (Trạng từ)    O    
2     được            Verb (Động từ)       O    
3     vượt            Verb (Động từ)       O    
4     xe              Noun (Danh từ)       O    
5     trong           Preposition (Giới từ) O    
6     trường_hợp      Noun (Danh từ)       O    
7     sau             Noun (Danh từ)       O    
8     đây             Pronoun (Đại từ)     O    
9     :               Punctuation (Dấu câu) O    
10    trên            Preposition (Giới từ) O    
11    cầu             Noun (Danh từ)       O    
12    hẹp             Adj